In [ ]:
# ============================================================
# Cell 1: ENVIRONMENT GATE — Colab A100 ONLY
# ============================================================
# MANDATORY: This cell must pass before ANY training.
# Refuses local execution. Verifies CUDA GPU.
import os, sys

_cwd = os.getcwd()
_is_local = _cwd.startswith("/Users/") or (_cwd.startswith("/home/") and "content" not in _cwd)

import torch
_has_cuda = torch.cuda.is_available()

if _is_local or not _has_cuda:
    print("=" * 65)
    print("  BLOCKED: This notebook must run on Google Colab with CUDA GPU")
    print(f"  Current dir : {_cwd}")
    print(f"  CUDA        : {_has_cuda}")
    print("=" * 65)
    print("\n  Runtime → Change runtime type → A100 GPU")
    raise SystemExit("Refusing local execution. Use Colab A100.")

# Passed — CUDA environment verified
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)

print(f"{'='*65}")
print(f"  ENVIRONMENT VERIFIED")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram_gb:.1f} GB")
print(f"  Compute cap  : {cc[0]}.{cc[1]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")
print(f"  Working dir  : {_cwd}")
print(f"{'='*65}")
os.system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

In [ ]:
# ============================================================
# Cell 2: Clone Repo + Install Dependencies
# ============================================================
import subprocess, os, sys, pathlib

REPO_URL = "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git"
PROJ_ROOT = "/content/nst"

if not os.path.isdir(os.path.join(PROJ_ROOT, "data")):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, PROJ_ROOT], check=True)
else:
    print("Repository already cloned. Pulling latest...")
    subprocess.run(["git", "-C", PROJ_ROOT, "pull", "--ff-only"], check=True)

os.chdir(PROJ_ROOT)
if PROJ_ROOT not in sys.path:
    sys.path.insert(0, PROJ_ROOT)

# Install package + deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJ_ROOT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets==2.21.0", "peft>=0.9", "transformers>=4.40",
                "accelerate", "sentencepiece", "protobuf"], check=True)

print(f"\nProject root: {PROJ_ROOT}")
print(f"Python: {sys.executable}")

# GPU auto-config
import torch
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)
supports_bf16 = cc >= (8, 0)

if supports_bf16:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# Determine batch config based on VRAM
if vram_gb >= 35:
    BS, GA = 32, 2
elif vram_gb >= 20:
    BS, GA = 24, 2
else:
    BS, GA = 16, 2

GPU_OVERRIDES = {
    "train": {
        "batch_size": BS,
        "grad_accum_steps": GA,
        "bf16": supports_bf16,
        "fp16": not supports_bf16,
        "tf32": supports_bf16,
        "fused_optimizer": True,
        "num_workers": 4,
    }
}

DEVICE = "cuda"
prec = "BF16" if supports_bf16 else "FP16"
print(f"\nGPU config: bs={BS}×{GA}={BS*GA} effective, {prec}, VRAM={vram_gb:.0f}GB")

In [ ]:
# ============================================================
# Cell 3: Verify datasets version compatibility
# ============================================================
import datasets
print(f"datasets version: {datasets.__version__}")
_ds_major = int(datasets.__version__.split(".")[0])
assert _ds_major < 3, (
    f"datasets {datasets.__version__} doesn't support FEVER loading scripts. "
    "Run: pip install datasets==2.21.0 then restart the kernel."
)
print("OK — compatible with FEVER loading script")

In [ ]:
# ============================================================
# Cell 4: Build Wiki Cache & Verify Evidence Quality
# ============================================================
# The FEVER dataset needs a wiki page cache to resolve evidence
# text from page title + sentence index. Without it, evidence is
# just page titles → NLI accuracy capped at ~60-70%.
import logging, os, time, sys
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Clear stale project modules for fresh imports
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

# ── Step 1: Build wiki cache if missing ──
from data.fever_wiki_cache import build_wiki_cache, cache_stats

cache_path = os.path.join(PROJ_ROOT, "data", "fever_wiki.db")
stats = cache_stats(cache_path)
if stats.get("exists"):
    print(f"Wiki cache exists: {stats['n_pages']} pages, {stats['size_mb']:.1f} MB")
else:
    print("Building wiki page cache (one-time, ~5-10 min)...")
    t0 = time.time()
    build_stats = build_wiki_cache(cache_path=cache_path)
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.0f}s: {build_stats['n_found']}/{build_stats['n_needed']} pages")

# ── Step 2: Load small data sample and verify evidence quality ──
from data.fever_dataset import load_fever_splits, print_fever_stats

splits_check = load_fever_splits(max_train=500, max_dev=200, dev_test_ratio=0.1, seed=42)
print_fever_stats(splits_check)

train_items = splits_check["train"]
n_with_evidence = sum(1 for it in train_items if len(it.get("gold_evidence_text", "")) > 30)
pct = 100 * n_with_evidence / max(1, len(train_items))
print(f"\nEvidence quality: {n_with_evidence}/{len(train_items)} ({pct:.0f}%) have >30 char evidence")

# Show sample evidence for manual verification
for i in range(min(5, len(train_items))):
    it = train_items[i]
    ev = it.get("gold_evidence_text", "")[:150]
    print(f"\n  [{i}] {it['label']}: {it['claim'][:80]}")
    print(f"       Evidence: {ev}")

if pct > 60:
    print(f"\n{'='*50}")
    print(f"  EVIDENCE CHECK PASSED — {pct:.0f}% coverage")
    print(f"{'='*50}")
elif pct > 20:
    print(f"\n  WARNING: Partial evidence ({pct:.0f}%). Results will be degraded.")
else:
    print(f"\n  CRITICAL: Only {pct:.0f}% have evidence. Wiki cache needed.")
    print(f"  Run: python main.py build-fever-wiki-cache")

In [ ]:
# ============================================================
# Cell 5: Smoke Test — 50 examples (validates pipeline end-to-end)
# ============================================================
import time, gc, json, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("Smoke test: 50 train / 25 dev / 1 epoch (VERI mode)")
print("Purpose: verify pipeline runs end-to-end on CUDA\n")

t0 = time.time()
from training.train_fever_veri import train_fever_veri

results_smoke = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides={
        "data": {"max_train": 50, "max_dev": 25, "dev_sample": 25},
        "train": {"epochs": 1, "eval_every_steps": 25, "patience": 100},
        "io": {"out_dir": "outputs_smoke_veri"},
    }
)
elapsed = time.time() - t0

dev = results_smoke.get("dev", {})
print(f"\nSmoke test complete in {elapsed:.1f}s")
print(f"  Dev acc: {dev.get('accuracy', 'N/A')}")
print(f"  (Accuracy is meaningless at 50 examples — this just validates the pipeline)")

# Verify CUDA was actually used
assert DEVICE == "cuda", "ERROR: Not running on CUDA!"
print(f"\n  Pipeline smoke test PASSED on {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Cell 6: 3K NEURAL BASELINE (Fair Comparison — Same Architecture)
# ============================================================
# DeBERTa-v3-large + LoRA, NO constraints.
# Establishes the ceiling that pure neural achieves on 3k.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_3k = train_fever_nst(
    "configs/fever_neural_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_neural_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_3k.json", "w") as f:
    json.dump(results_neural_3k, f, indent=2, default=str)
print(f"\n  Saved to results_neural_3k.json")

In [ ]:
# ============================================================
# Cell 7: 3K NST-VERI (Constraint-Enhanced Training — THE CRITICAL RUN)
# ============================================================
# DeBERTa-v3-large + LoRA + verification heads + contrastive + adaptive lambda
# 3-phase: NLI+aux → +contrastive → +constraints
#
# WATCH FOR:
#   - constraint_loss > 0 (constraints must be active)
#   - fire_rate > 0 (constraints must fire)
#   - mean_lambda > 0.1 (constraints must have weight)
#   - dev_acc >= neural baseline (constraints must not hurt)
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NST-VERI: Verification-Enhanced Constraint Training")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_3k = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# ── Constraint activity analysis ──
train_log = results_veri_3k.get("train_log", [])
if train_log:
    phase3_entries = [e for e in train_log if e.get("phase", 0) >= 3]
    if phase3_entries:
        cst_losses = [e.get("loss_constraint", 0) for e in phase3_entries]
        lambdas = [e.get("mean_lambda", 0) for e in phase3_entries]
        print(f"\n  CONSTRAINT DIAGNOSTICS:")
        print(f"    Phase 3 entries   : {len(phase3_entries)}")
        print(f"    Constraint loss   : min={min(cst_losses):.4f} max={max(cst_losses):.4f} mean={sum(cst_losses)/len(cst_losses):.4f}")
        print(f"    Mean lambda       : min={min(lambdas):.4f} max={max(lambdas):.4f} mean={sum(lambdas)/len(lambdas):.4f}")
        if max(cst_losses) > 0.001:
            print(f"    CONSTRAINTS ARE ACTIVE")
        else:
            print(f"    WARNING: CONSTRAINTS STILL INACTIVE")
    else:
        print(f"\n  WARNING: No Phase 3 entries in training log")

# Constraint calibration
calib = results_veri_3k.get("constraint_calibration", {})
if calib:
    print(f"\n  CONSTRAINT CALIBRATION (on dev):")
    for cname, cstats in calib.items():
        print(f"    {cname}: precision={cstats.get('precision', 0):.3f} fire_rate={cstats.get('fire_rate', 0):.3f}")

with open("results_veri_3k.json", "w") as f:
    json.dump(results_veri_3k, f, indent=2, default=str)
print(f"\n  Saved to results_veri_3k.json")

In [ ]:
# ============================================================
# Cell 8: 3K COMPARISON — Neural vs NST-VERI
# ============================================================
import json, os

experiments = {}
for name, path in [("neural_3k", "results_neural_3k.json"),
                   ("veri_3k", "results_veri_3k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*65}")
print(f"  3K COMPARISON: Neural vs NST-VERI")
print(f"{'='*65}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8}")
print(f"  {'─'*55}")

for name, r in experiments.items():
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "N/A")
    best = r.get("best_dev_acc", "N/A")
    ece = dev.get("ece", "N/A")
    acc_s = f"{acc:.4f}" if isinstance(acc, (int, float)) else str(acc)
    best_s = f"{best:.4f}" if isinstance(best, (int, float)) else str(best)
    ece_s = f"{ece:.4f}" if isinstance(ece, (int, float)) else str(ece)
    print(f"  {name:<25} {acc_s:>10} {best_s:>10} {ece_s:>8}")

# ── Signal assessment ──
if "neural_3k" in experiments and "veri_3k" in experiments:
    n_acc = experiments["neural_3k"].get("best_dev_acc", 0)
    v_acc = experiments["veri_3k"].get("best_dev_acc", 0)
    delta = v_acc - n_acc
    print(f"\n  Delta (VERI - Neural): {delta:+.4f}")
    if delta > 0.01:
        print(f"  SIGNAL: NST-VERI shows improvement. Full run justified.")
    elif delta > -0.01:
        print(f"  NEUTRAL: No clear signal yet. May need config tuning.")
    else:
        print(f"  WARNING: NST-VERI underperforms. Investigate before full run.")

In [ ]:
# ============================================================
# Cell 9: FULL NST-VERI RUN (Only after 3k shows signal)
# ============================================================
# Run ONLY after Cell 8 comparison shows VERI >= Neural on 3k.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NST-VERI: DeBERTa-v3-large + LoRA + Verification")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_full = train_fever_veri(
    "configs/fever_gold_nst_veri_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_full.get("dev", {})
print(f"\n{'='*65}")
print(f"  FULL NST-VERI RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_full.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Constraint diagnostics
train_log = results_veri_full.get("train_log", [])
phase3 = [e for e in train_log if e.get("phase", 0) >= 3]
if phase3:
    cst = [e.get("loss_constraint", 0) for e in phase3]
    lam = [e.get("mean_lambda", 0) for e in phase3]
    print(f"\n  Constraint loss: {min(cst):.4f}—{max(cst):.4f} (mean {sum(cst)/len(cst):.4f})")
    print(f"  Mean lambda:     {min(lam):.4f}—{max(lam):.4f} (mean {sum(lam)/len(lam):.4f})")

# Held-out dev_test
dt = results_veri_full.get("dev_test")
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")

with open("results_veri_full.json", "w") as f:
    json.dump(results_veri_full, f, indent=2, default=str)
print(f"\n  Saved to results_veri_full.json")

In [ ]:
# ============================================================
# Cell 10: FULL NEURAL BASELINE (Fair Comparison)
# ============================================================
# Same DeBERTa-v3-large + LoRA, NO constraints.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NEURAL BASELINE: DeBERTa-v3-large + LoRA")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_full = train_fever_nst(
    "configs/fever_gold_neural.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_full.get("dev", {})
print(f"\n{'='*65}")
print(f"  FULL NEURAL RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_neural_full.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_full.json", "w") as f:
    json.dump(results_neural_full, f, indent=2, default=str)
print(f"\n  Saved to results_neural_full.json")

In [ ]:
# ============================================================
# Cell 11: FINAL COMPARISON & HONEST REPORT
# ============================================================
import json, os, glob

experiments = {}
for name, path in [
    ("neural_3k", "results_neural_3k.json"),
    ("veri_3k", "results_veri_3k.json"),
    ("neural_full", "results_neural_full.json"),
    ("veri_full", "results_veri_full.json"),
]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*65}")
print(f"  FINAL RESULTS — FEVER Gold Evidence Label Accuracy")
print(f"{'='*65}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8} {'Brier':>8}")
print(f"  {'─'*63}")

for name, r in sorted(experiments.items()):
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "?")
    best = r.get("best_dev_acc", "?")
    ece = dev.get("ece", "?")
    brier = dev.get("brier", "?")
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
    print(f"  {name:<25} {fmt(acc):>10} {fmt(best):>10} {fmt(ece):>8} {fmt(brier):>8}")

# Per-label breakdown for full runs
for name in ["neural_full", "veri_full"]:
    if name in experiments:
        dev = experiments[name].get("dev", {})
        print(f"\n  {name} per-label:")
        for label, stats in dev.get("per_label", {}).items():
            print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Held-out dev_test comparison
for name in ["neural_full", "veri_full"]:
    if name in experiments:
        dt = experiments[name].get("dev_test")
        if dt:
            print(f"\n  {name} held-out dev_test: {dt.get('accuracy', '?')}")

print(f"\n{'='*65}")
print(f"  HONEST ASSESSMENT")
print(f"{'='*65}")
if "veri_full" in experiments:
    final_acc = experiments["veri_full"].get("best_dev_acc", 0)
    if final_acc >= 0.90:
        print(f"  TARGET REACHED: {final_acc:.4f} >= 0.90")
    else:
        print(f"  TARGET NOT YET REACHED: {final_acc:.4f} < 0.90")
        print(f"  Next steps: analyze per-label failures, tune constraints, retrain")

In [ ]:
# ============================================================
# Cell 12: SEED SWEEP (Reproducibility — 3 seeds)
# ============================================================
# Run after achieving 90%+ on seed=42 to verify result is real.
import time, json, gc, sys

seeds = [42, 43, 44]
seed_results = {}

for seed in seeds:
    for mod in list(sys.modules.keys()):
        if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                            "logic.", "symbolic.", "retrieval."]):
            del sys.modules[mod]
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n{'='*50}")
    print(f"  Seed {seed} — Full NST-VERI")
    print(f"{'='*50}")

    t0 = time.time()
    from training.train_fever_veri import train_fever_veri
    r = train_fever_veri(
        "configs/fever_gold_nst_veri_a100.yaml",
        config_overrides={
            **GPU_OVERRIDES,
            "seed": seed,
            "io": {"out_dir": f"outputs_veri_seed{seed}"},
        },
    )
    elapsed = time.time() - t0
    seed_results[seed] = r
    dev = r.get("dev", {})
    print(f"  Seed {seed}: acc={dev.get('accuracy','?')} best={r.get('best_dev_acc','?')} ({elapsed/60:.1f}min)")

# Summary
accs = [seed_results[s].get("best_dev_acc", 0) for s in seeds]
import numpy as np
print(f"\n{'='*50}")
print(f"  SEED SWEEP RESULTS")
print(f"  Accs: {accs}")
print(f"  Mean: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"{'='*50}")

In [ ]:
# ============================================================
# Cell 13: SAVE ARTIFACTS — Download from Colab
# ============================================================
import json, os, shutil, glob

# Gather all result files
artifacts = glob.glob("results_*.json") + glob.glob("outputs_*/report.json")
print(f"Result artifacts: {artifacts}")

# Create a zip for easy download
artifact_dir = "nst_results"
os.makedirs(artifact_dir, exist_ok=True)
for f in artifacts:
    shutil.copy(f, artifact_dir)
# Add configs used
for cfg in glob.glob("configs/fever_*3k*.yaml") + glob.glob("configs/fever_gold_nst_veri*.yaml"):
    shutil.copy(cfg, artifact_dir)

shutil.make_archive("nst_results", "zip", ".", artifact_dir)
print(f"Download: nst_results.zip")

# Colab download helper
try:
    from google.colab import files
    files.download("nst_results.zip")
except ImportError:
    print("Not on Colab — download nst_results.zip manually")

In [ ]:
# ============================================================
# Cell 14: LEAKAGE AUDIT — Verify no data contamination
# ============================================================
# Run AFTER training to verify results are honest.
import json, sys, os

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_dataset import load_fever_splits

# Load same splits with same seed
splits = load_fever_splits(max_train=None, max_dev=None, dev_test_ratio=0.1, seed=42)

train_claims = {it["claim"] for it in splits["train"]}
dev_claims = {it["claim"] for it in splits["dev"]}
dev_test_claims = {it["claim"] for it in splits.get("dev_test", [])}

# Check overlaps
train_dev_overlap = train_claims & dev_claims
train_devtest_overlap = train_claims & dev_test_claims
dev_devtest_overlap = dev_claims & dev_test_claims

print(f"{'='*50}")
print(f"  LEAKAGE AUDIT")
print(f"{'='*50}")
print(f"  Train size     : {len(splits['train'])}")
print(f"  Dev size       : {len(splits['dev'])}")
print(f"  Dev-test size  : {len(splits.get('dev_test', []))}")
print(f"  Train∩Dev      : {len(train_dev_overlap)} overlapping claims")
print(f"  Train∩DevTest  : {len(train_devtest_overlap)} overlapping claims")
print(f"  Dev∩DevTest    : {len(dev_devtest_overlap)} overlapping claims")

if len(train_dev_overlap) == 0 and len(train_devtest_overlap) == 0:
    print(f"\n  LEAKAGE CHECK PASSED — No contamination detected")
else:
    print(f"\n  WARNING: Potential leakage detected!")
    if train_dev_overlap:
        print(f"    Example overlap: {list(train_dev_overlap)[:3]}")